# 03 — Demand Forecasting
Train-test split → Baseline → Seasonal Naive → Prophet. Compare MAE & MAPE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('../data/processed/processed_sales.csv')
df['date'] = pd.to_datetime(df['date'])

# Aggregate across all stores for item 1
sku_df = (df[df['item'] == 1]
          .groupby('date', as_index=False)['sales']
          .sum()
          .sort_values('date'))

train = sku_df.iloc[:-90].copy()
test  = sku_df.iloc[-90:].copy().reset_index(drop=True)

print('Train:', train.shape, '| Test:', test.shape)

## Model 1 — Baseline (7-day Rolling Mean)

In [ ]:
last_7_avg = train['sales'].tail(7).mean()
test['baseline'] = last_7_avg

mae_baseline  = mean_absolute_error(test['sales'], test['baseline'])
mape_baseline = np.mean(np.abs((test['sales'] - test['baseline']) / test['sales'])) * 100

print(f'Baseline  → MAE: {mae_baseline:.2f} | MAPE: {mape_baseline:.1f}%')

## Model 2 — Seasonal Naive (7-day lag)

In [ ]:
seasonal_naive = []
for i in range(len(test)):
    idx = -(7 - (i % 7))
    seasonal_naive.append(train['sales'].iloc[idx])

test['seasonal_naive'] = seasonal_naive

mae_seasonal  = mean_absolute_error(test['sales'], test['seasonal_naive'])
mape_seasonal = np.mean(np.abs((test['sales'] - test['seasonal_naive']) / test['sales'])) * 100

print(f'Seasonal Naive → MAE: {mae_seasonal:.2f} | MAPE: {mape_seasonal:.1f}%')

## Model 3 — Prophet

In [ ]:
# pip install prophet  (run in terminal if not installed)
from prophet import Prophet

prophet_df = train[['date','sales']].rename(columns={'date':'ds','sales':'y'})

model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
model.fit(prophet_df)

future   = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

test['prophet'] = forecast.tail(90)['yhat'].values

mae_prophet  = mean_absolute_error(test['sales'], test['prophet'])
mape_prophet = np.mean(np.abs((test['sales'] - test['prophet']) / test['sales'])) * 100

print(f'Prophet → MAE: {mae_prophet:.2f} | MAPE: {mape_prophet:.1f}%')

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model':  ['Baseline', 'Seasonal Naive', 'Prophet'],
    'MAE':    [mae_baseline,  mae_seasonal,  mae_prophet],
    'MAPE %': [mape_baseline, mape_seasonal, mape_prophet],
})
print(comparison.to_string(index=False))

plt.figure(figsize=(12, 4))
plt.plot(test['date'], test['sales'],         label='Actual')
plt.plot(test['date'], test['baseline'],      label='Baseline',       linestyle='--')
plt.plot(test['date'], test['seasonal_naive'],label='Seasonal Naive', linestyle='--')
plt.plot(test['date'], test['prophet'],       label='Prophet',        linestyle='--')
plt.legend()
plt.title('Forecast Model Comparison — Item 1 (90-day test)')
plt.tight_layout()
plt.show()